# Blog figures — Anyscale post (BLOG_ANYSCALE_DRAFT.md)

Infra-story figures. Same conventions as the personal notebook: `$FM_BASE`,
loud skips, PNG+SVG into the repo's `figures/` dir (commit them). Data figures load the same canonical
artifacts so the two posts can't drift apart.

| cell | asset | source |
|---|---|---|
| arch | **A1 / B2-annotated** architecture hero (stage -> scale knob + hardware) | drawn here |
| results | **A2** fulltest results strip (the one benchmark figure this post needs) | fulltest CI jsons |
| knobs | **A3** laptop -> cluster is a config change | `configs/*.yaml`, parsed live |
| distxgb | **A4** distributed-XGBoost story (3 tries + success vs single-node band) | `downstream/xl_fulltest_distxgb*/` |
| stages | **A5** GPUs per pipeline stage (0 -> 8 autoscale story) | `job_full.yaml` facts |
| velocity | **A6** commit-velocity histogram | `git log` of this checkout (labeled) |
| costs | **A7 / B15** cost per run | `RUN_COSTS` — paste from console |

Note (audit): the draft's `distributed_xgb.py` code block is from the research
branch — either check the script in or reframe "every code block is copied
from the template" before publish.


In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
# Figures default into the REPO (templates/fintech_transaction_fm/figures/) so
# they can be committed and embedded in the blog posts directly; FM_FIG_OUT
# overrides. burst_stats.json / run_costs.json ledgers travel with them.
_repo_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
FIG_OUT = os.environ.get("FM_FIG_OUT", os.path.join(_repo_root, "figures"))
os.makedirs(FIG_OUT, exist_ok=True)

# Palette (validated): one accent for the story (our embedding), muted ink for
# every other model, an ordinal blue ramp for the three context lengths.
# Highlight-one-gray-the-rest; selective direct labels on the accent series.
ACCENT, MUTED = "#2a78d6", "#898781"
INK, INK2 = "#0b0b0b", "#52514e"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CTX_RAMP = {512: "#86b6ef", 1024: "#2a78d6", 2048: "#104281"}

SCALES = [("full", 512), ("xl", 1024), ("xxl", 2048)]
NV_BASELINE, NV_FUSION = 0.1238, 0.1755   # their published points

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.labelcolor": INK2, "font.size": 10.5, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

def save(fig, name):
    for ext, kw in (("png", {"dpi": 200}), ("svg", {})):
        fig.savefig(f"{FIG_OUT}/{name}.{ext}", bbox_inches="tight", **kw)
    print(f"[saved] {FIG_OUT}/{name}.png + .svg")

def load_results(path):
    if not os.path.exists(path):
        print(f"[SKIP] missing {path}")
        return None
    return json.load(open(path))["results"]

ci100k = {sc: load_results(f"{BASE}/downstream/{sc}/bootstrap_ci.json") for sc, _ in SCALES}
# Table-1 2048 row = the 20-epoch eval (same training budget as 512/1024);
# downstream/xxl holds the 40-ep continuation rerun (table 2's model).
ci100k["xxl"] = (load_results(f"{BASE}/downstream/xxl_old_1783532341/bootstrap_ci.json")
                 or ci100k["xxl"])
cifull = {sc: load_results(f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json") for sc, _ in SCALES}

## A1 — architecture hero (stage \u2192 scale knob + hardware)\n\nThe anchor figure of the post.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

STAGES = [
    # (engine, title, scale knob, hardware)
    ("Ray Data",  "merchant vocab\n(freq scan, top-K)",      "one column read;\nmerchant_top_k",        "CPU workers"),
    ("Ray Data",  "tokenize\n(static/dynamic, map_groups)",  "seq_len 512\u21922048;\nshuffle_partitions", "CPU-only nodes\n(GPU nodes CPU:0)"),
    ("Ray Train", "masked pretrain + InfoNCE\n(PyTorch DDP)", "num_workers \u00d7 GPU;\nepochs, batch",  "4\u00d7 A10G\n(g5.xlarge)"),
    ("Ray Data",  "embedding extraction\n(CPU read \u2192 GPU forward)", "num_workers=8\n(pure inference)", "0\u21928 GPUs\nautoscaled"),
    ("XGBoost",   "fraud readout\nraw-13 vs embedding",      "xgboost==3.2.0 pin;\nCPU or CUDA",        "head node"),
    ("rank",      "next-merchant reco\n(HR@K / NDCG@K)",     "same frozen backbone,\nInfoNCE head",     "head node"),
    ("Ray Serve", "online embed + score\n(cached)",          "autoscaling\nreplicas",                   "1 GPU replica"),
]
ENGINE_C = {"Ray Data": ACCENT, "Ray Train": "#104281", "XGBoost": "#8a6d1f", "rank": "#8a6d1f", "Ray Serve": "#3d7a4e"}
ARTIFACTS = ["raw parquet\n+ splits.json", "vocab.json", "token windows\n(object store)",
             "checkpoint", "embeddings\nparquet", "metrics +\ntest preds", "", ""]

def draw_arch(annotated: bool, name: str):
    n = len(STAGES)
    fig, ax = plt.subplots(figsize=(14.5, 4.6 if annotated else 3.4))
    ax.set_xlim(-0.75, n - 0.25); ax.set_ylim(-1.9 if annotated else -1.25, 1.35)
    ax.axis("off")
    ax.annotate("raw transactions\n(Parquet, S3)", (-0.62, 0), ha="center", va="center",
                fontsize=9, color=INK2, style="italic")
    for i, (engine, title, knob, hw) in enumerate(STAGES):
        c = ENGINE_C[engine]
        box = FancyBboxPatch((i - 0.36, -0.42), 0.72, 0.9,
                             boxstyle="round,pad=0.02,rounding_size=0.06",
                             fc=SURFACE, ec=c, lw=1.6, zorder=3)
        ax.add_patch(box)
        ax.annotate(engine, (i, 0.30), ha="center", fontsize=9.5, color=c, weight="bold", zorder=4)
        ax.annotate(title, (i, -0.08), ha="center", fontsize=8.2, color=INK, zorder=4)
        if i:  # arrow from previous stage
            ax.add_patch(FancyArrowPatch((i - 1 + 0.40, 0.05), (i - 0.40, 0.05),
                                         arrowstyle="-|>", mutation_scale=13, color=INK2, lw=1.3, zorder=2))
        if ARTIFACTS[i]:  # durable artifact under the boundary
            ax.annotate(ARTIFACTS[i], (i - 0.5 if i else i - 0.62, 0.78), ha="center",
                        fontsize=7.4, color=INK2, zorder=4,
                        bbox=dict(boxstyle="round,pad=0.25", fc=GRID, ec="none", alpha=0.7))
        if annotated:
            ax.annotate(f"scale knob\n{knob}", (i, -0.95), ha="center", va="top", fontsize=7.6, color=INK2)
            ax.annotate(hw, (i, -1.55), ha="center", va="top", fontsize=7.6, color=c)
    ax.annotate("same code laptop \u2192 multi-node: you change the config, not the program",
                (0.5, 0.985), xycoords="axes fraction", ha="center", fontsize=10, color=INK)
    save(fig, name)
    plt.show()

draw_arch(annotated=True, name="b2_architecture_anyscale")


## A2 — the one results figure (fulltest, CI-separated)

In [ ]:
rows = []
for sc, ctx in SCALES:
    r = cifull.get(sc)
    if r:
        lab = f"embed\u2192XGB \u00b7 {ctx}" + (" (40ep)" if ctx == 2048 else "")
        rows.append((lab, r.get("embed_xgb"), True))
base_r = cifull["full"] and cifull["full"].get("baseline")
if rows and base_r:
    fig, ax = plt.subplots(figsize=(8, 4.4))
    ax.axhspan(*base_r["ap_ci95"], color=GRID, alpha=0.55, zorder=1)
    ax.axhline(base_r["ap"], color=INK2, lw=1.0, zorder=2)
    ax.annotate(f'raw-13 baseline {base_r["ap"]:.3f}', (0.985, base_r["ap"]),
                xycoords=("axes fraction", "data"), ha="right", va="bottom",
                fontsize=9, color=INK2)
    for x, (lab, r, ours) in enumerate(rows):
        lo, hi = r["ap_ci95"]
        ax.errorbar(x, r["ap"], yerr=[[r["ap"] - lo], [hi - r["ap"]]], fmt="o",
                    color=ACCENT, ecolor=ACCENT, ms=7, capsize=3, lw=1.5, zorder=3)
        ax.annotate(f'{r["ap"]:.3f}', (x, hi), textcoords="offset points",
                    xytext=(0, 5), ha="center", fontsize=9.5, color=INK)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([l for l, _, _ in rows], fontsize=9.5)
    ax.set_xlim(-0.6, len(rows) - 0.4)
    ax.set_ylabel("Average precision (full 2.44M-row test)")
    ax.set_title("The embedding beats the identical-rows raw baseline, CI-separated",
                 fontsize=11.5, color=INK, pad=10)
    fig.subplots_adjust(bottom=0.14)
    fig.text(0.5, 0.02, "Dots = point AP, whiskers = bootstrap 95% CI, band = baseline 95% CI. Identical test rows.",
             ha="center", fontsize=8.8, color=INK2)
    save(fig, "a2_results")
    plt.show()
else:
    print("[SKIP] A2 — need fulltest bootstrap_ci.json for all scales")


## A3 — laptop \u2192 cluster is a config change\n\nParsed live from `configs/*.yaml`, so the figure can never disagree with the repo.

In [ ]:
import yaml

KNOB_ROWS = [
    ("cards sampled",        lambda c: f"{c['data']['num_cards']:,}"),
    ("context (txns/window)", lambda c: f"{c['tokenize']['seq_len']:,}"),
    ("d_model / layers",     lambda c: f"{c['model']['d_model']} / {c['model']['n_layers']}"),
    ("pretrain epochs",      lambda c: str(c['pretrain']['epochs'])),
    ("train workers x GPU",  lambda c: f"{c['pretrain']['num_workers']} x " + ("GPU" if c['pretrain']['use_gpu'] else "CPU")),
    ("embed workers (job_full\noverrides to 8)", lambda c: f"{c['embed']['num_workers']}" + (" (GPU)" if c['embed']['use_gpu'] else " (CPU)")),
]
SHOW = ["smoke", "small", "full", "xl", "xxl"]
cfg_dir = os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd(), "configs")
cfgs = {}
for s in SHOW:
    p = os.path.join(cfg_dir, f"{s}.yaml")
    if os.path.exists(p):
        cfgs[s] = yaml.safe_load(open(p))
if cfgs:
    cols = [s for s in SHOW if s in cfgs]
    cell_text = [[fn(cfgs[s]) for s in cols] for _, fn in KNOB_ROWS]
    fig, ax = plt.subplots(figsize=(1.9 * len(cols) + 2.4, 0.5 * len(KNOB_ROWS) + 1.4))
    ax.axis("off")
    tbl = ax.table(cellText=cell_text, rowLabels=[r for r, _ in KNOB_ROWS],
                   colLabels=[c + ("  (headline)" if c == "full" else "") for c in cols],
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor(GRID)
        if r == 0: cell.set_text_props(color=INK, weight="bold"); cell.set_facecolor(SURFACE)
        if c == -1: cell.set_text_props(color=INK2, ha="right")
    ax.set_title("Same program at every scale \u2014 one YAML per column is the ENTIRE diff",
                 fontsize=12, color=INK, pad=18)
    save(fig, "a3_scale_knobs")
    plt.show()
else:
    print(f"[SKIP] A3 — configs/ not found near {os.getcwd()}")


## A4 — distributed XGBoost: the fusion-at-scale story\n\nAll four ledgered runs from `downstream/xl_fulltest_distxgb*/`; single-node CI band as the bar to clear.

In [ ]:
DIST_RUNS = [  # (dir suffix, label)
    ("xl_fulltest_distxgb_old_1783719521", "4 workers \u00b7 hist (first try)"),
    ("xl_fulltest_distxgb_ctrl1w",         "1 worker control \u00b7 hist"),
    ("xl_fulltest_distxgb",                "4 workers \u00b7 hist"),
    ("xl_fulltest_distxgb_gpu",            "4 workers \u00b7 GPU"),
]
runs = []
for d, lab in DIST_RUNS:
    p = f"{BASE}/downstream/{d}/distributed_xgb_metrics.json"
    if os.path.exists(p):
        m = json.load(open(p))
        runs.append((lab, m["ap"], m.get("num_workers"), m.get("best_iteration")))
    else:
        print(f"[skip run] {p}")
ref = cifull.get("xl") and cifull["xl"].get("embed_xgb")  # single-node 1024 fulltest
if runs and ref:
    fig, ax = plt.subplots(figsize=(9, 4.4))
    ax.axhspan(*ref["ap_ci95"], color=GRID, alpha=0.6, zorder=1)
    ax.axhline(ref["ap"], color=INK2, lw=1.0, zorder=2)
    ax.annotate(f'single-node embed\u2192XGB {ref["ap"]:.3f} (95% CI band)',
                (0.02, ref["ap_ci95"][1]), xycoords=("axes fraction", "data"),
                va="bottom", fontsize=9, color=INK2)
    for x, (lab, ap, nw, bi) in enumerate(runs):
        ours = "GPU" in lab
        c = ACCENT if ours else MUTED
        ax.plot([x], [ap], "o", ms=9, color=c, zorder=3)
        ax.annotate(f"{ap:.3f}", (x, ap), textcoords="offset points", xytext=(0, 9),
                    ha="center", fontsize=9.5, color=INK)
    ax.set_xticks(range(len(runs)))
    ax.set_xticklabels([r[0] for r in runs], fontsize=9)
    ax.set_xlim(-0.6, len(runs) - 0.4)
    ax.set_ylabel("Average precision (full 2.44M-row test)")
    ax.set_title("Distributed XGBoost on the 1024 embedding: three tries to match "
                 "the single-node result \u2014 the GPU run lands inside its CI",
                 fontsize=11.5, color=INK, pad=10)
    save(fig, "a4_distributed_xgb")
    plt.show()
else:
    print("[SKIP] A4 — need xl_fulltest_distxgb*/distributed_xgb_metrics.json + xl fulltest CI")


## A5 — GPUs follow the pipeline (the 0\u21928 autoscale story)

In [ ]:
STAGE_GPUS = [  # from job_full.yaml / configs/full.yaml
    ("data +\nvocab", 0), ("tokenize", 0), ("pretrain", 4), ("embed", 8),
    ("XGBoost +\nreco", 0), ("serve", 1),
]
fig, ax = plt.subplots(figsize=(9, 3.8))
xs = np.arange(len(STAGE_GPUS))
ax.step(np.append(xs, len(xs)) - 0.5, [g for _, g in STAGE_GPUS] + [STAGE_GPUS[-1][1]],
        where="post", color=ACCENT, lw=2.2, zorder=3)
ax.fill_between(np.append(xs, len(xs)) - 0.5, [g for _, g in STAGE_GPUS] + [STAGE_GPUS[-1][1]],
                step="post", color=ACCENT, alpha=0.12, zorder=2)
for x, (lab, g) in enumerate(STAGE_GPUS):
    ax.annotate(str(g), (x, g), textcoords="offset points", xytext=(0, 6),
                ha="center", fontsize=10, color=INK, weight="bold")
ax.set_xticks(xs); ax.set_xticklabels([l for l, _ in STAGE_GPUS], fontsize=9.5)
ax.set_yticks([0, 4, 8]); ax.set_ylim(-0.4, 9.6); ax.set_xlim(-0.5, len(xs) - 0.5)
ax.set_ylabel("A10G GPUs held")
ax.set_title("GPU nodes scale 0 \u2192 8 \u2192 0 with the pipeline \u2014 min_nodes: 0, "
             "CPU stages can't land on them (CPU: 0)", fontsize=11.5, color=INK, pad=10)
save(fig, "a5_stage_gpus")
plt.show()


## A6 — commit velocity, ledgered\n\nThe draft's 161/115/52/45 stats came from memory and don't match this branch — this cell prints git's actual numbers; update the draft to whatever it says.

In [ ]:
import subprocess
try:
    out = subprocess.run(["git", "log", "--pretty=%ad", "--date=short", "--", "."],
                         capture_output=True, text=True, cwd=os.path.dirname(os.getcwd())
                         if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
    dates = pd.Series(out.stdout.split())
    branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
except Exception as e:
    dates = pd.Series(dtype=str); branch = "?"
if len(dates):
    counts = dates.value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(9, 3.6))
    xs = np.arange(len(counts))
    ax.bar(xs, counts.values, color=ACCENT, width=0.7, edgecolor=SURFACE, linewidth=1.5)
    for x, v in zip(xs, counts.values):
        ax.annotate(str(v), (x, v), textcoords="offset points", xytext=(0, 3),
                    ha="center", fontsize=9, color=INK)
    ax.set_xticks(xs); ax.set_xticklabels(counts.index, fontsize=8.5, rotation=30, ha="right")
    ax.set_ylabel("commits touching this template")
    ax.set_title(f"Campaign velocity \u2014 commits/day on `{branch}` (ledgered from git, "
                 "not memory)", fontsize=11.5, color=INK, pad=10)
    fig.subplots_adjust(bottom=0.22)
    save(fig, "a6_commit_velocity")
    plt.show()
    print(f"total commits: {counts.sum()}  |  use THESE numbers in the velocity section, "
          "not the remembered 161/115/52/45")
else:
    print("[SKIP] A6 — no git history available here")


## B15 — cost per run

Job costs are not queryable from inside the workspace — pull them from the
Anyscale console (job page \u2192 cost) into `RUN_COSTS` below, then run the cell.
Job IDs from BLOG_ASSETS.md. Skips loudly while any cost is None; once filled
it exports the cost figure and a `run_costs.json` ledger the drafts can cite
(replacing the "$15\u201325 per headline run" placeholder).


In [ ]:
RUN_COSTS = [
    # (label, prodjob id, USD from console — FILL ME)
    ("512 full run (e2e)",        "<console: fintech-transaction-fm-full>",  None),
    ("1024 run (e2e)",            "prodjob_1nxzlze72xgv7imvgvlw6hk9eb",      None),
    ("2048 run (e2e)",            "prodjob_szkm6hlic3hkhgrn4admydltua",      None),
    ("2048 20\u219240ep continuation", "<console: job_xxl_continue>",       None),
    ("pinned+CUDA eval",          "prodjob_n8tv97crr4nasnsjmw1ne3x2yg",      None),
    ("fulltest evals (3\u00d7)", "<console: job_fulltest*>",                None),
]
missing = [l for l, _, c in RUN_COSTS if c is None]
if missing:
    print(f"[SKIP] B15 — fill costs from the console for: {missing}")
else:
    with open(f"{FIG_OUT}/run_costs.json", "w") as f:
        json.dump([{"run": l, "job_id": j, "usd": c} for l, j, c in RUN_COSTS], f, indent=2)
    labels = [l for l, _, _ in RUN_COSTS]; costs = [c for _, _, c in RUN_COSTS]
    fig, ax = plt.subplots(figsize=(8.5, 3.6))
    ys = np.arange(len(labels))[::-1]
    bars = ax.barh(ys, costs, height=0.62, color=ACCENT, edgecolor=SURFACE, linewidth=2)
    for y, c in zip(ys, costs):
        ax.annotate(f"${c:,.0f}", (c, y), textcoords="offset points", xytext=(5, 0),
                    va="center", fontsize=9.5, color=INK)
    ax.set_yticks(ys); ax.set_yticklabels(labels, fontsize=9.5)
    ax.set_xlabel("cost (USD, Anyscale console)"); ax.grid(axis="y", visible=False)
    ax.set_title(f"What the campaign actually cost \u2014 total ${sum(costs):,.0f}",
                 fontsize=11.5, color=INK, pad=10)
    save(fig, "b15_costs")
    plt.show()


---\n**Produced here:** `b2_architecture_anyscale`, `a2_results`, `a3_scale_knobs`, `a4_distributed_xgb`, `a5_stage_gpus`, `a6_commit_velocity`, `b15_costs` (once `RUN_COSTS` is filled).\n\n**Manual:** hero-graphic design pass (Cowork/Figma, using A1 as the content spec), console costs.